# Prepare MAGFiLO archive for ChatGPT analysis

Open this notebook in **Google Colab**, select **Runtime → Run all**, and authorize Google Drive using the SAME Google account that owns the dataset. It will automatically search for `filament-segmentation-2026.zip`, then split it into four ~180 MiB pieces in the same folder. The original archive remains untouched. The pieces and checksum manifest stay private in your Google Drive.

If the archive cannot be found, the notebook will list visible top-level folders and explain how to fix the account/mount mismatch. After `Finished`, return to ChatGPT.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mount is ready.')


In [ ]:
from pathlib import Path
from hashlib import sha256
import json
import math
import os
import unicodedata

ARCHIVE_NAME = "filament-segmentation-2026.zip"
EXPECTED_BYTES = 703574877  # Exact size previously verified from your connected Google Drive
CHUNK_BYTES = 180 * 1024 * 1024  # Below the 256-MiB connector limit
BUF_BYTES = 8 * 1024 * 1024

roots = [Path(p) for p in ("/content/drive/MyDrive", "/content/drive/My Drive") if Path(p).is_dir()]
if not roots:
    raise RuntimeError(
        "Google Drive is not mounted here. Open this notebook in Google Colab, "
        "run the FIRST cell (drive.mount), approve access, and then run this cell."
    )

# First search common places, including variations of the Arabic folder's Unicode spelling.
candidates = []
for root in roots:
    candidates.extend(root.glob(ARCHIVE_NAME))
    candidates.extend(root.glob("*/" + ARCHIVE_NAME))

# If needed, inspect nested folders, but keep traversal bounded to avoid a long Drive scan.
if not candidates:
    for root in roots:
        visited = 0
        for folder, subfolders, files in os.walk(root):
            visited += 1
            depth = len(Path(folder).relative_to(root).parts)
            subfolders[:] = [d for d in subfolders if not d.startswith(".")]
            if depth >= 5 or visited > 12000:
                subfolders[:] = []
            if ARCHIVE_NAME in files:
                candidates.append(Path(folder) / ARCHIVE_NAME)
            if visited > 12000:
                break

# Prefer the archive whose size exactly matches the file previously found in Drive.
candidates = [p for p in candidates if p.is_file()]
archive = next((p for p in candidates if p.stat().st_size == EXPECTED_BYTES), None)
if archive is None and len(candidates) == 1:
    archive = candidates[0]

if archive is None:
    print("Mounted roots:", [str(p) for p in roots])
    for root in roots:
        print(f"Folders visible at {root}:",
              [x.name for x in root.iterdir() if x.is_dir()][:35])
    if candidates:
        print("Matching filenames at unexpected sizes:", [(str(p), p.stat().st_size) for p in candidates])
    raise FileNotFoundError(
        "The ZIP is not visible in the mounted Drive account. "
        "The source is in folder 'بيانات سولر' in the connected Google account. "
        "Make sure Colab is mounted to THAT SAME Google account, "
        "and that the ZIP is under My Drive (not Shared with me). "
        "If needed, add a shortcut to that folder in My Drive, then remount."
    )

print("Found archive:", archive)
print("Size:", f"{archive.stat().st_size / 1048576:.1f} MiB")


In [ ]:
source_size = archive.stat().st_size
n_parts = math.ceil(source_size / CHUNK_BYTES)
if n_parts < 1:
    raise ValueError("The archive is empty.")

parts = []
whole_digest = sha256()
with archive.open("rb") as source:
    for index in range(1, n_parts + 1):
        part = archive.with_name(f"{archive.name}.part{index:02d}")
        temporary = archive.with_name(f"{archive.name}.part{index:02d}.tmp")
        expected = min(CHUNK_BYTES, source_size - (index - 1) * CHUNK_BYTES)
        remaining = expected
        digest = sha256()
        try:
            with temporary.open("wb") as destination:
                while remaining > 0:
                    block = source.read(min(BUF_BYTES, remaining))
                    if not block:
                        raise IOError("The original ZIP ended unexpectedly.")
                    destination.write(block)
                    digest.update(block)
                    whole_digest.update(block)
                    remaining -= len(block)
            os.replace(temporary, part)
        finally:
            if temporary.exists():
                temporary.unlink()
        parts.append({"name": part.name, "bytes": expected, "sha256": digest.hexdigest()})
        print(f"{index}/{n_parts}: {part.name} written and verified", flush=True)

manifest = {
    "original_file": archive.name,
    "original_size": source_size,
    "original_sha256": whole_digest.hexdigest(),
    "parts": parts,
    "chunk_size": CHUNK_BYTES,
}
manifest_path = archive.with_name("filament-segmentation-2026.parts.json")
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nFinished: {len(parts)} parts + manifest saved in {archive.parent}")
